<div align="center" style="
    background: linear-gradient(135deg, #667eea, #764ba2);
    padding:35px;
    border-radius:15px;
    box-shadow:0 8px 25px rgba(0,0,0,0.25);
">
  <h1 style="
      color:white;
      font-family: 'Segoe UI', sans-serif;
      margin:0;
      font-size:42px;
      letter-spacing:1px;
  ">
    Recommendation System
  </h1>
</div>

<h3 style="
    text-align:left;
    font-family: 'Georgia', serif;
    font-weight:600;
    letter-spacing:1.5px;
    color:#2c3e50;
">
    🏠 99 Acres by Prince Kumar
</h3>

# Importing Libaries

In [2]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
pd.set_option('display.max_rows',None)
pd.set_option('display.max_columns',None)

# Loading Dataset

In [4]:
df = pd.read_csv(r"F:\ML files\99_Acers\real_estate_data - real_estate_data (1).csv")

In [5]:
df.shape

(247, 7)

In [6]:
df.head()

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Salon', 'Restaurant', 'Spa'..."
1,M3M Crown,"3, 4 BHK Apartment in Sector 111, Gurgaon","['DPSG Palam Vihar Gurugram', 'The NorthCap Un...","{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N...",https://www.99acres.com/m3m-crown-sector-111-g...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Bowling Alley', 'Mini Theatre', 'Manicured G..."
2,Adani Brahma Samsara Vilasa,"Land, 3, 4 BHK Independent Floor in Sector 63,...","['AIPL Business Club Sector 62', 'Heritage Xpe...","{'AIPL Business Club Sector 62': '2.7 Km', 'He...",https://www.99acres.com/adani-brahma-samsara-v...,{'3 BHK': {'building_type': 'Independent Floor...,"['Terrace Garden', 'Gazebo', 'Fountain', 'Amph..."
3,Sobha City,"2, 3, 4 BHK Apartment in Sector 108, Gurgaon","['The Shikshiyan School', 'WTC Plaza', 'Luxus ...","{'The Shikshiyan School': '2.9 KM', 'WTC Plaza...",https://www.99acres.com/sobha-city-sector-108-...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Volley Ball Court', 'Aerobi..."
4,Signature Global City 93,"2, 3 BHK Independent Floor in Sector 93 Gurgaon","['Pranavananda Int. School', 'DLF Site central...","{'Pranavananda Int. School': '450 m', 'DLF Sit...",https://www.99acres.com/signature-global-city-...,{'2 BHK': {'building_type': 'Independent Floor...,"['Mini Theatre', 'Doctor on Call', 'Concierge ..."


In [7]:
df.isnull().sum()

PropertyName          0
PropertySubName       0
NearbyLocations       0
LocationAdvantages    0
Link                  0
PriceDetails          0
TopFacilities         0
dtype: int64

In [8]:
df['PriceDetails'].head()

0    {'2 BHK': {'building_type': 'Apartment', 'area...
1    {'3 BHK': {'building_type': 'Apartment', 'area...
2    {'3 BHK': {'building_type': 'Independent Floor...
3    {'2 BHK': {'building_type': 'Apartment', 'area...
4    {'2 BHK': {'building_type': 'Independent Floor...
Name: PriceDetails, dtype: object

In [9]:
df['TopFacilities'].head()

0    ['Swimming Pool', 'Salon', 'Restaurant', 'Spa'...
1    ['Bowling Alley', 'Mini Theatre', 'Manicured G...
2    ['Terrace Garden', 'Gazebo', 'Fountain', 'Amph...
3    ['Swimming Pool', 'Volley Ball Court', 'Aerobi...
4    ['Mini Theatre', 'Doctor on Call', 'Concierge ...
Name: TopFacilities, dtype: object

In [ ]:
# Recemendation Syestem based on LocationAdvantages

In [10]:
df_loc = df[['PropertyName','LocationAdvantages','Link']]

In [11]:
df_loc.head()

,PropertyName,LocationAdvantages,Link
0,Smartworld One DXP,"{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...
1,M3M Crown,"{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N...",https://www.99acres.com/m3m-crown-sector-111-g...
2,Adani Brahma Samsara Vilasa,"{'AIPL Business Club Sector 62': '2.7 Km', 'He...",https://www.99acres.com/adani-brahma-samsara-v...
3,Sobha City,"{'The Shikshiyan School': '2.9 KM', 'WTC Plaza...",https://www.99acres.com/sobha-city-sector-108-...
4,Signature Global City 93,"{'Pranavananda Int. School': '450 m', 'DLF Sit...",https://www.99acres.com/signature-global-city-...


In [12]:
# df_loc['LocationAdvantages'] = df_loc['LocationAdvantages'].str.lower()

# Converting String to Dict

In [13]:
import ast

def parse_dict(x):
    try:
        if isinstance(x, str):
            return ast.literal_eval(x)
        else:
            return {}
    except:
        return {}

df_loc['LocationAdvantages'] = df_loc['LocationAdvantages'].apply(parse_dict)

In [91]:
print(type(df_loc['LocationAdvantages'].iloc[0]))

<class 'dict'>


# Strandardizeing Distance

In [92]:
def convert_distance(value):

    try:
        value = value.lower().strip()

        if 'km' in value:
            return float(value.replace('km','').strip())

        elif 'meter' in value:
            return float(value.replace('meter','').strip()) / 1000

        elif 'm' in value:
            return float(value.replace('m','').strip()) / 1000

        else:
            return float(value)

    except:
        return None

In [93]:
df_loc['LocationAdvantages'] = df_loc['LocationAdvantages'].apply(
    lambda d: {k: convert_distance(v) for k,v in d.items()} if isinstance(d, dict) else {}
)

In [94]:
df_loc.head(5)

,PropertyName,LocationAdvantages
0,Smartworld One DXP,"{'Bajghera Road': 0.8, 'Palam Vihar Halt': 2.5..."
1,M3M Crown,"{'DPSG Palam Vihar Gurugram': 1.4, 'The NorthC..."
2,Adani Brahma Samsara Vilasa,"{'AIPL Business Club Sector 62': 2.7, 'Heritag..."
3,Sobha City,"{'The Shikshiyan School': 2.9, 'WTC Plaza': 4...."
4,Signature Global City 93,"{'Pranavananda Int. School': 0.45, 'DLF Site c..."


In [99]:
df_loc['LocationAdvantages'] = df_loc['LocationAdvantages'].apply(
    lambda d: {k.lower(): v for k,v in d.items()} if isinstance(d, dict) else {}
)

In [100]:
def search_properties(location, radius):

    results = []

    for _, row in df_loc.iterrows():

        loc_dict = row['LocationAdvantages']

        if location in loc_dict:

            if loc_dict[location] <= radius:
                results.append(row['PropertyName'])

    return results

In [110]:
search_properties("dwarka expy", 0.5)

['M3M Capital']

In [161]:
df_loc.to_csv('df_loc.csv')

In [104]:
unique_locations = set()

for loc_dict in df_loc['LocationAdvantages']:
    
    if isinstance(loc_dict, dict):
        unique_locations.update(loc_dict.keys())



In [1]:
# unique_locations

In [108]:
unique_locations = sorted(unique_locations)

In [107]:
len(unique_locations)

1026

# Recemendation Syestem based on TopFacilities

In [15]:
df_top = df[['PropertyName','TopFacilities','Link']]

In [6]:
df_top.head()

,PropertyName,TopFacilities,Link
0,Smartworld One DXP,"['Swimming Pool', 'Salon', 'Restaurant', 'Spa'...",https://www.99acres.com/smartworld-one-dxp-sec...
1,M3M Crown,"['Bowling Alley', 'Mini Theatre', 'Manicured G...",https://www.99acres.com/m3m-crown-sector-111-g...
2,Adani Brahma Samsara Vilasa,"['Terrace Garden', 'Gazebo', 'Fountain', 'Amph...",https://www.99acres.com/adani-brahma-samsara-v...
3,Sobha City,"['Swimming Pool', 'Volley Ball Court', 'Aerobi...",https://www.99acres.com/sobha-city-sector-108-...
4,Signature Global City 93,"['Mini Theatre', 'Doctor on Call', 'Concierge ...",https://www.99acres.com/signature-global-city-...


In [7]:
df_top.to_csv('df_top.csv')

# Converting String to List 

In [16]:
def parse_facilities(x):
    try:
        if isinstance(x, str):
            return ast.literal_eval(x)
        else:
            return []
    except:
        return []

df_top['TopFacilities'] = df_top['TopFacilities'].apply(parse_facilities)

In [142]:
type(df_top['TopFacilities'].iloc[0])

list

In [17]:
df_top['TopFacilities'] = df_top['TopFacilities'].apply(
    lambda x: [i.lower() for i in x]
)

In [18]:
df_top['facilities_text'] = df_top['TopFacilities'].apply(lambda x: " ".join(x))

In [145]:
all_facilities = set()

for fac_list in df_top['TopFacilities']:
    if isinstance(fac_list, list):
        all_facilities.update(fac_list)

all_facilities = sorted(all_facilities)

In [146]:

facility_matrix = pd.DataFrame(0, index=df_top.index, columns=all_facilities)

for i, facilities in enumerate(df_top['TopFacilities']):
    for f in facilities:
        if f in facility_matrix.columns:
            facility_matrix.loc[i, f] = 1

## Computing Cosine Similarity

In [147]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(facility_matrix)

## Recommendation Function 

In [148]:
def recommend(property_name):

    idx = df_top[df_top['PropertyName'] == property_name].index[0]

    scores = list(enumerate(similarity[idx]))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:6]

    indices = [i[0] for i in scores]

    return df_top['PropertyName'].iloc[indices]

In [154]:
recommend("M3M Natura")

0                Smartworld One DXP
57                DLF The Magnolias
118          Pioneer Urban Presidia
157            Emaar MGF Palm Hills
21     Krisumi Waterfall Residences
Name: PropertyName, dtype: object

In [153]:
facility_matrix.shape

(247, 110)

In [156]:
import pickle

In [157]:
pickle.dump(similarity, open("similarity.pkl","wb"))